In [2]:
import pandas as pd
import sqlite3
import os



csv_path = 'data/g31_Equipment_Operators_Final.csv'
db_path = 'data/equipment_management.db'

df = pd.read_csv(csv_path)
conn= sqlite3.connect(db_path)

equipment = df[['equipment_id', 'name', 'creation_date']].drop_duplicates()
equipment = equipment.rename(columns={'equipment_id': 'id'})
equipment.to_sql('Equipment', conn, if_exists='replace', index=False)


operators = df[['operator_id', 'title', 'category', 'birth_date']].drop_duplicates()
operators = operators.rename(columns={'operator_id': 'id'})
operators.to_sql('Operator', conn, if_exists='replace', index=False)

m_types = df[['maintenance_type_id', 'equipment_id']].dropna().drop_duplicates()
m_types = m_types.rename(columns={'maintenance_type_id': 'id'})
m_types['id'] = m_types['id'].astype(int)
m_types.to_sql('MaintenanceType', conn, if_exists='replace', index=False)


events = df[['maintenance_event_id', 'equipment_id', 'maintenance_type_id', 'Maintenance_Date', 'extra_info']].dropna(subset=['maintenance_event_id'])
events = events.rename(columns={'maintenance_event_id': 'id'})
for col in ['id', 'equipment_id', 'maintenance_type_id']:
    events[col] = events[col].astype(int)
events.to_sql('MaintenanceEvent', conn, if_exists='replace', index=False)

usage = df[['equipment_id', 'operator_id', 'utilization_date', 'cost']]
usage.to_sql('Equipment_Operator', conn, if_exists='replace', index=True, index_label='id')

print("All tables created successfully:")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

conn.close()

All tables created successfully:
[('Equipment',), ('Operator',), ('MaintenanceType',), ('MaintenanceEvent',), ('Equipment_Operator',)]
